# Lab 5 — look before you change anything

The file: `data/raw/env_wasmun.tsv`, municipal waste. The pipeline (`uv run python pipeline.py`) never reads this
notebook: it is where you look. **No AI for the first 20 minutes of the lab.** Look at the values with your own eyes.

Five questions. Two are supplied: run them and read them. Two are the lecture's queries, on this file. The fifth
is yours, from an empty cell. Paste each query and what it returned into `DIAGNOSIS.md`, part 3, **before** you
change `scripts/clean.py`: a fix erases the evidence.

In [ ]:
import os
from pathlib import Path

import duckdb
import pandas as pd

pd.set_option("display.max_rows", 400)   # show every row of a result: a census is never "the top few"

# Anchor to the project folder (DS1, Block 1), then stand there: every path below is from the project folder.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)
print("Working in:", Path.cwd())   # must print this project's folder

con = duckdb.connect()             # an in-memory database; it reads the files in data/raw/ directly

### Supplied — the colleague's first two steps, so you can query `long` here

This is sections A and B of `scripts/clean.py`: the file as text, the packed column split, the years turned into rows.
One row of `long` is one observation: one series in one year. `cell` is the text Eurostat wrote.

In [ ]:
con.sql("""
    CREATE OR REPLACE VIEW long AS
    SELECT split_part(series, ',', 1) AS freq,
           split_part(series, ',', 2) AS wst_oper,
           split_part(series, ',', 3) AS unit,
           split_part(series, ',', 4) AS geo,
           CAST(year AS INTEGER)      AS year,
           cell
    FROM read_csv('data/raw/env_wasmun.tsv', delim = '\t', all_varchar = true, names = ['series'])
    UNPIVOT INCLUDE NULLS (cell FOR year IN (COLUMNS('^[0-9]{4}$')))
""")
con.sql("SELECT COUNT(*) AS observations FROM long").df()

## Look

#### 1. Supplied: run it and read it. What did Eurostat write in Germany's cell, which the report left blank for 2024? (waste generated per inhabitant: `wst_oper = 'GEN'`, `unit = 'KG_HAB'`)

In [ ]:
con.sql("""
    SELECT geo, year, cell
    FROM long
    WHERE wst_oper = 'GEN' AND unit = 'KG_HAB' AND geo = 'DE' AND year IN (2010, 2024)
""").df()

#### 2. The census the lecture ran, on this file: what follows the space in a cell, and how often?

In [ ]:
# your query

#### 3. What did the colleague's cast, `TRY_CAST(cell AS DOUBLE)`, swallow? Count the cells where Eurostat wrote a number and the cast returned NULL — in the whole file, and in the unit silver keeps.

In [ ]:
# your query

#### 4. Supplied: run it and read it. The census of `unit`: which units, and how many observations each? Read it against the `reason` in section C.

In [ ]:
con.sql("""
    SELECT unit, COUNT(*) AS n
    FROM long
    GROUP BY unit
""").df()

#### 5. Yours, from the empty cell: no slide shows this query. The report leaves 22 of the 37 countries blank for 2024. A blank is right when Eurostat wrote `:` (not available). It is wrong when Eurostat wrote a number and the cast lost it. Take the report's cells — waste generated (`GEN`), `KG_HAB`, 2024, the countries (not `EU27_2020`) — and split them three ways, in one query: the cast read a value; Eurostat wrote `:`; Eurostat wrote a number the cast lost.

In [ ]:
# your query